# Préparation de la base  de données 

## 1. Imports + check environnement

In [1]:
import os, re, json, shutil, math, time
from pathlib import Path

import numpy as np
import pandas as pd

# Optional: OpenCV (recommandé)
try:
    import cv2
    HAS_CV2 = True
except Exception as e:
    HAS_CV2 = False
    print("⚠️ cv2 non disponible. Certaines étapes (détection visage) seront désactivées.", e)

print("HAS_CV2 =", HAS_CV2)


HAS_CV2 = True


## 2. Configuration chemins + paramètres

In [2]:
# === A MODIFIER ===
RAW_ROOT = Path(r"C:/Users/khodj/Documents/M2_ISI/PFE/Affectnet2")   # dossier brut (kaggle)
OUT_ROOT = Path(r"C:/Users/khodj/Documents/M2_ISI/PFE/data_clean")       # dossier de sortie standardisé

# Dossiers de sortie
OUT_IMG_DIR = OUT_ROOT / "images"          # PNG/JPG recadrées
OUT_META_DIR = OUT_ROOT / "meta"           # csv/json logs
OUT_TRASH_DIR = OUT_ROOT / "trash"         # fichiers corrompus etc.

for d in [OUT_IMG_DIR, OUT_META_DIR, OUT_TRASH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Paramètres preprocessing
TARGET_SIZE = (224, 224)     # (W, H) standard pour vision
SAVE_EXT = ".png"            # .png conseillé
MAX_IMAGES = None            # ex: 5000 pour tester, sinon None
RANDOM_SEED = 42

# Détection visage: simple + robuste offline (Haar cascade)
USE_FACE_DETECTION = True    # passe à False si tu veux juste resize sans recadrage visage
FACE_MARGIN = 0.25           # marge autour du visage recadré

print("RAW_ROOT:", RAW_ROOT)
print("OUT_ROOT:", OUT_ROOT)


RAW_ROOT: C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2
OUT_ROOT: C:\Users\khodj\Documents\M2_ISI\PFE\data_clean


## 3. Fonctions utilitaires (scan images, hash, nettoyage)

In [3]:
from hashlib import md5

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def file_md5(path: Path, chunk_size=1024*1024) -> str:
    h = md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def safe_relpath(p: Path, root: Path) -> str:
    try:
        return str(p.relative_to(root))
    except Exception:
        return str(p)

def list_images(root: Path):
    imgs = []
    for p in root.rglob("*"):
        if p.is_file() and is_image_file(p):
            imgs.append(p)
    return imgs


## 4. Détection visage (Haar cascade offline) + fallback

In [4]:
if HAS_CV2:
    # Haar cascade face detector (disponible avec OpenCV)
    haar_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    face_cascade = cv2.CascadeClassifier(haar_path)
    if face_cascade.empty():
        print("⚠️ Haar cascade introuvable. Désactive USE_FACE_DETECTION ou réinstalle opencv-data.")
else:
    face_cascade = None

def detect_face_bbox(img_bgr):
    """
    Retourne bbox (x,y,w,h) du meilleur visage ou None.
    """
    if not HAS_CV2 or face_cascade is None or face_cascade.empty():
        return None
    
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    if len(faces) == 0:
        return None
    
    # Choisir le plus grand visage (aire max)
    faces = sorted(faces, key=lambda b: b[2]*b[3], reverse=True)
    return faces[0]

def crop_with_margin(img_bgr, bbox, margin=0.25):
    h, w = img_bgr.shape[:2]
    x, y, bw, bh = bbox
    mx = int(bw * margin)
    my = int(bh * margin)
    x1 = max(0, x - mx)
    y1 = max(0, y - my)
    x2 = min(w, x + bw + mx)
    y2 = min(h, y + bh + my)
    return img_bgr[y1:y2, x1:x2], (x1, y1, x2-x1, y2-y1)


## 5. Détection du format de la  bdd (CSV ou dossiers)

In [5]:
def find_annotation_csv(root: Path):
    # Cherche un CSV probable d'annotations
    candidates = []
    for p in root.rglob("*.csv"):
        name = p.name.lower()
        if any(k in name for k in ["labels"]):
            candidates.append(p)
    # On prend le plus gros (souvent le bon)
    if not candidates:
        return None
    candidates = sorted(candidates, key=lambda x: x.stat().st_size, reverse=True)
    return candidates[0]

ann_csv = find_annotation_csv(RAW_ROOT)
print("CSV annotations détecté:", ann_csv)

all_imgs = list_images(RAW_ROOT)
print("Nb images trouvées:", len(all_imgs))
print("Exemple:", safe_relpath(all_imgs[0], RAW_ROOT) if all_imgs else "Aucune image")


CSV annotations détecté: C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2\labels.csv
Nb images trouvées: 31002
Exemple: anger\image0000006.jpg


## 6. Pour une bdd en dossier 

In [6]:
# Mapping d'émotions (adaptable)
EMO_MAP = {
    "neutral": "neutral",
    "happy": "happy",
    "sad": "sad",
    "surprise": "surprise",
    "fear": "fear",
    "anger": "anger",
    "disgust": "disgust",
    "contempt": "contempt",
}

def label_from_path(p: Path) -> str:
    parts = [s.lower() for s in p.parts]
    for k in EMO_MAP.keys():
        if k in parts:
            return EMO_MAP[k]
    return "unknown"

# Crée une table brute (même sans CSV)
df_raw = pd.DataFrame({
    "src_path": [str(p) for p in all_imgs],
})
df_raw["rel_path"] = df_raw["src_path"].apply(lambda s: safe_relpath(Path(s), RAW_ROOT))
df_raw["emotion"] = df_raw["src_path"].apply(lambda s: label_from_path(Path(s)))

df_raw.head()


,src_path,rel_path,emotion
0,C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2...,anger\image0000006.jpg,anger
1,C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2...,anger\image0000007.jpg,anger
2,C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2...,anger\image0000012.jpg,anger
3,C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2...,anger\image0000035.jpg,anger
4,C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2...,anger\image0000060.jpg,anger


## 7. Un csv d'annotations 

In [7]:
def guess_cols(df: pd.DataFrame):
    cols = {c.lower(): c for c in df.columns}
    # candidates
    path_candidates = [c for c in df.columns if re.search(r"(path|file|image|img)", c, re.I)]
    emo_candidates  = [c for c in df.columns if re.search(r"(emotion|label|exp)", c, re.I)]
    val_candidates  = [c for c in df.columns if re.search(r"(valence)", c, re.I)]
    aro_candidates  = [c for c in df.columns if re.search(r"(arousal)", c, re.I)]
    bbox_candidates = [c for c in df.columns if re.search(r"(bbox|x|y|w|h)", c, re.I)]
    return path_candidates, emo_candidates, val_candidates, aro_candidates, bbox_candidates

if ann_csv is not None:
    df_ann = pd.read_csv(ann_csv)
    print("CSV shape:", df_ann.shape)
    path_c, emo_c, val_c, aro_c, bbox_c = guess_cols(df_ann)
    print("Path candidates:", path_c[:10])
    print("Emotion candidates:", emo_c[:10])
    print("Valence candidates:", val_c[:10])
    print("Arousal candidates:", aro_c[:10])

    # Choix automatique (premier candidat)
    path_col = "pth"
    emo_col  = emo_c[0] if emo_c else None
    val_col  = val_c[0] if val_c else None
    aro_col  = aro_c[0] if aro_c else None

    if path_col is None:
        raise ValueError("Impossible d'identifier la colonne chemin image dans le CSV. Donne-moi les colonnes et je te l’adapte.")

    df_raw = df_ann.copy()
    df_raw.rename(columns={path_col: "rel_path"}, inplace=True)
    if emo_col:
        df_raw.rename(columns={emo_col: "emotion"}, inplace=True)
    else:
        df_raw["emotion"] = "unknown"

    if val_col and "valence" not in df_raw.columns:
        df_raw.rename(columns={val_col: "valence"}, inplace=True)
    if aro_col and "arousal" not in df_raw.columns:
        df_raw.rename(columns={aro_col: "arousal"}, inplace=True)

    # Normaliser rel_path (certains CSV ont juste un nom de fichier)
    df_raw["rel_path"] = df_raw["rel_path"].astype(str).str.replace("\\", "/", regex=False)
    df_raw.head()
else:
    print("Pas de CSV -> utilise la cellule 6A (labels depuis dossiers).")


CSV shape: (31002, 2)
Path candidates: []
Emotion candidates: ['label']
Valence candidates: []
Arousal candidates: []


## 8. Construction des chemins absolus + filtrage des manquants

In [8]:
# Convertit rel_path en src_path absolu
def to_abs_path(rel_path: str) -> Path:
    p = Path(rel_path)
    # si le CSV contient un chemin relatif déjà complet
    cand = RAW_ROOT / p
    if cand.exists():
        return cand
    # sinon, on tente de retrouver par nom de fichier (fallback)
    name = p.name
    matches = [x for x in all_imgs if x.name == name]
    return matches[0] if matches else cand  # cand peut ne pas exister

df_raw["src_path"] = df_raw["rel_path"].apply(lambda s: str(to_abs_path(s)))

# Filtre existants
df_raw["exists"] = df_raw["src_path"].apply(lambda s: Path(s).exists())
missing = df_raw[~df_raw["exists"]]
print("Manquants:", len(missing))

df = df_raw[df_raw["exists"]].copy()

# Option: limiter pour tester
if MAX_IMAGES is not None:
    df = df.sample(n=min(MAX_IMAGES, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)

print("Images retenues:", len(df))
df.head()


Manquants: 0
Images retenues: 31002


,rel_path,emotion,src_path,exists
0,anger/image0000006.jpg,surprise,C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2...,True
1,anger/image0000007.jpg,anger,C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2...,True
2,anger/image0000012.jpg,anger,C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2...,True
3,anger/image0000035.jpg,fear,C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2...,True
4,anger/image0000060.jpg,anger,C:\Users\khodj\Documents\M2_ISI\PFE\Affectnet2...,True


## 9. Normalisation des labels 

In [9]:
def normalize_emotion(x):
    if pd.isna(x):
        return "unknown"
    s = str(x).strip().lower()
    s = s.replace(" ", "_")
    # si numeric labels -> à adapter selon source
    if s.isdigit():
        return s  # on garde brut si on ne sait pas le mapping
    # harmoniser quelques variantes
    aliases = {
        "surprised": "surprise",
        "fearful": "fear",
        "angry": "anger",
    }
    s = aliases.get(s, s)
    return s

df["emotion"] = df["emotion"].apply(normalize_emotion)

print(df["emotion"].value_counts().head(20))


emotion
surprise    4889
happy       4382
anger       4160
disgust     3776
fear        3753
contempt    3588
sad         3352
neutral     3102
Name: count, dtype: int64


## 10. Fonction “process one image” (lecture, détection, crop, resize, save)

In [10]:
def read_image_bgr(path: Path):
    if not HAS_CV2:
        return None, "no_cv2"
    img = cv2.imread(str(path))
    if img is None:
        return None, "imread_failed"
    return img, None

def save_png(img_bgr, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    ok = cv2.imwrite(str(out_path), img_bgr)
    return ok

def process_row(row, idx:int):
    src = Path(row["src_path"])
    emo = row.get("emotion", "unknown")

    img, err = read_image_bgr(src)
    if err:
        return {"ok": False, "reason": err, "src_path": str(src), "emotion": emo, "out_path": None}

    h, w = img.shape[:2]
    if h < 32 or w < 32:
        return {"ok": False, "reason": "too_small", "src_path": str(src), "emotion": emo, "out_path": None}

    face_bbox = None
    crop_bbox = None

    if USE_FACE_DETECTION:
        face_bbox = detect_face_bbox(img)
        if face_bbox is not None:
            img_crop, crop_bbox = crop_with_margin(img, face_bbox, margin=FACE_MARGIN)
        else:
            # fallback: garder l'image entière si pas de visage (ou tu peux rejeter)
            img_crop = img
    else:
        img_crop = img

    # Resize
    if HAS_CV2:
        img_res = cv2.resize(img_crop, TARGET_SIZE, interpolation=cv2.INTER_AREA)
    else:
        return {"ok": False, "reason": "no_cv2", "src_path": str(src), "emotion": emo, "out_path": None}

    # Nom de fichier stable
    # -> basé sur md5 du chemin relatif + index
    rel = safe_relpath(src, RAW_ROOT)
    uid = md5(rel.encode("utf-8")).hexdigest()[:16]
    out_name = f"affectnet_{uid}_{idx:07d}{SAVE_EXT}"
    out_path = OUT_IMG_DIR / out_name

    ok = save_png(img_res, out_path)
    if not ok:
        return {"ok": False, "reason": "imwrite_failed", "src_path": str(src), "emotion": emo, "out_path": None}

    return {
        "ok": True,
        "reason": None,
        "id": out_name.replace(SAVE_EXT, ""),
        "src_path": str(src),
        "rel_path": rel,
        "emotion": emo,
        "orig_w": int(w),
        "orig_h": int(h),
        "face_bbox": face_bbox,
        "crop_bbox": crop_bbox,
        "out_path": str(out_path),
    }


## 11. Boucle de traitement (avec reprise) + logs

In [11]:
from tqdm import tqdm

# Fichiers de sortie
META_OUT = OUT_META_DIR / "labels_vision.csv"
LOG_OUT  = OUT_META_DIR / "preprocess_log.csv"

# Reprise si déjà traité
already = set()
if META_OUT.exists():
    old = pd.read_csv(META_OUT)
    already = set(old["src_path"].astype(str).tolist())
    print("Reprise: déjà traités =", len(already))

results = []
fails = []

for i, row in tqdm(list(df.iterrows()), total=len(df)):
    if str(row["src_path"]) in already:
        continue

    r = process_row(row, idx=i)
    if r["ok"]:
        results.append(r)
    else:
        fails.append(r)

print("Nouveaux OK:", len(results))
print("Nouveaux FAIL:", len(fails))

df_ok = pd.DataFrame(results)
df_fail = pd.DataFrame(fails)

# Append propre
if META_OUT.exists() and len(df_ok) > 0:
    df_ok.to_csv(META_OUT, mode="a", header=False, index=False)
elif len(df_ok) > 0:
    df_ok.to_csv(META_OUT, index=False)

if len(df_fail) > 0:
    if LOG_OUT.exists():
        df_fail.to_csv(LOG_OUT, mode="a", header=False, index=False)
    else:
        df_fail.to_csv(LOG_OUT, index=False)

print("✅ Sauvegardé:", META_OUT)
print("⚠️ Logs:", LOG_OUT)


100%|██████████| 31002/31002 [40:09<00:00, 12.87it/s]  


Nouveaux OK: 31002
Nouveaux FAIL: 0
✅ Sauvegardé: C:\Users\khodj\Documents\M2_ISI\PFE\data_clean\meta\labels_vision.csv
⚠️ Logs: C:\Users\khodj\Documents\M2_ISI\PFE\data_clean\meta\preprocess_log.csv


## 12. Post-clean : suppression doublons + sanity checks

In [12]:
labels = pd.read_csv(META_OUT) if META_OUT.exists() else pd.DataFrame()
print("Total labels:", len(labels))

# Doublons par src_path
before = len(labels)
labels = labels.drop_duplicates(subset=["src_path"], keep="first").reset_index(drop=True)
after = len(labels)
print("Doublons supprimés:", before - after)

labels.to_csv(META_OUT, index=False)

# Check fichiers manquants côté sortie
missing_out = labels[~labels["out_path"].apply(lambda s: Path(s).exists())]
print("Sorties manquantes:", len(missing_out))

labels["emotion"].value_counts().head(20)


Total labels: 31002
Doublons supprimés: 0
Sorties manquantes: 0


emotion
surprise    4889
happy       4382
anger       4160
disgust     3776
fear        3753
contempt    3588
sad         3352
neutral     3102
Name: count, dtype: int64

## 13. Génération du fichier Json qui résume la base 

In [13]:
# CSV final minimal MindVoice
final_csv = OUT_META_DIR / "mindvoice_vision_labels.csv"
cols = ["id", "out_path", "emotion", "orig_w", "orig_h", "rel_path"]
keep = [c for c in cols if c in labels.columns]
labels[keep].to_csv(final_csv, index=False)
print("✅ CSV final:", final_csv)

# JSON metadata (optionnel, utile pour rapport)
final_json = OUT_META_DIR / "mindvoice_vision_summary.json"
summary = {
    "source": "AffectNet subset (Kaggle)",
    "target_size": list(TARGET_SIZE),
    "use_face_detection": USE_FACE_DETECTION,
    "face_margin": FACE_MARGIN,
    "n_samples": int(len(labels)),
    "emotion_counts": labels["emotion"].value_counts().to_dict() if "emotion" in labels.columns else {},
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
with open(final_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("✅ Summary JSON:", final_json)


✅ CSV final: C:\Users\khodj\Documents\M2_ISI\PFE\data_clean\meta\mindvoice_vision_labels.csv
✅ Summary JSON: C:\Users\khodj\Documents\M2_ISI\PFE\data_clean\meta\mindvoice_vision_summary.json


## 14. Train test split

In [14]:
labels = pd.read_csv(META_OUT)

rng = np.random.default_rng(RANDOM_SEED)
idx = np.arange(len(labels))
rng.shuffle(idx)

val_ratio = 0.15
n_val = int(len(labels) * val_ratio)

val_idx = idx[:n_val]
train_idx = idx[n_val:]

labels["split"] = "train"
labels.loc[val_idx, "split"] = "val"

split_csv = OUT_META_DIR / "mindvoice_vision_splits.csv"
labels[["id", "out_path", "emotion", "split"]].to_csv(split_csv, index=False)
print("✅ Splits:", split_csv)
print(labels["split"].value_counts())


✅ Splits: C:\Users\khodj\Documents\M2_ISI\PFE\data_clean\meta\mindvoice_vision_splits.csv
split
train    26352
val       4650
Name: count, dtype: int64
